# MISFIT + nnFoundation GPU Test

An end-to-end check of MISFIT with both openly released nnFoundation models on a
real GPU and real CT data ([MSD](http://medicaldecathlon.com/) Task 9, Spleen).
It answers:

1. **Memory and speed.** Which batch sizes fit at each patch size, and how fast
   do they train?
2. **Loading.** Do `nnFoundationCNN` (`resenc`) and `nnFoundationViT` (`primus-x`)
   load and beat the naive baseline and a random network on real scans?
3. **Pretraining.** Does `misfit_train --init-weights` run, and how does it
   compare with the same run from random weights?
4. **Resume, downstream commands, and MIST.** Do `--resume`, `misfit_inspect`,
   `misfit_encode`, `misfit_embed`, and the transfer into MIST all work?

Use a **G4 or A100** runtime (`Runtime → Change runtime type`), then
`Runtime → Run all`. The last cell prints a results block to copy.

The spleen run is a functional test, not a benchmark: it has few volumes, a very
short schedule, and only reconstruction metrics.

## Settings

`MISFIT_SOURCE` is what `pip` installs. To test unpushed code, build a wheel with
`python -m build`, upload it to this runtime, and set this to its path.

In [ ]:
MISFIT_SOURCE = "git+https://github.com/aecelaya/MISFIT@main"

# Masking matches how each checkpoint was pretrained (nnssl's trainers).
MODELS = {
    "nnFoundationCNN": {
        "model": "resenc",
        "url": "https://huggingface.co/MIC-DKFZ/nnFoundationCNN/resolve/main/checkpoint_final.pth",
        "patch": 192,
        "mask_patch_size": 16,
        "mask_ratio": 0.75,
    },
    "nnFoundationViT": {
        "model": "primus-x",
        "url": "https://huggingface.co/MIC-DKFZ/nnFoundationViT/resolve/main/checkpoint_final.pth",
        "patch": 192,
        "mask_patch_size": 8,
        "mask_ratio": 0.8,
    },
}

# Each training run below is EPOCHS x STEPS_PER_EPOCH optimizer steps.
EPOCHS = 5
STEPS_PER_EPOCH = 50

# (model, patch) pairs for the memory and speed probe, at each batch size.
PROBE = [
    ("resenc", 128),
    ("resenc", 192),
    ("primus-medium", 96),
    ("primus-medium", 192),
    ("primus-x", 96),
    ("primus-x", 128),
    ("primus-x", 192),
]
PROBE_BATCH_SIZES = [1, 2, 4]
PROBE_STEPS = 5

# Largest batch size to use for training; the probe may lower it.
MAX_TRAIN_BATCH_SIZE = 2

In [ ]:
!pip install -q "{MISFIT_SOURCE}"
# Reinstall so a rerun picks up new commits; the version number may not change.
!pip install -q --force-reinstall --no-deps "{MISFIT_SOURCE}"

## Environment and helpers

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
import time
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch

import misfit
import misfit.models  # noqa: F401
from misfit.models.model_registry import get_model_from_registry
from misfit.utils import hardware

WORK = Path("misfit_test")
WORK.mkdir(exist_ok=True)
INDEX = WORK / "index.parquet"
CPU_WORKERS = min(8, os.cpu_count() or 1)

results = {"checks": [], "probe": [], "evaluate": [], "training": {}, "seconds": {}}


def check(name, passed, detail=""):
    results["checks"].append({"name": name, "passed": bool(passed), "detail": str(detail)})
    print(f"{'PASS' if passed else 'FAIL'}  {name}  {detail}")


def run(label, command, *args):
    """Run a MISFIT command with live output, and record its result and time."""
    cmd = [sys.executable, "-m", f"misfit.cli.{command}_entrypoint", *map(str, args)]
    print(f"\n>>> {label}")
    start = time.time()
    tail = deque(maxlen=30)
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            print(line, end="")
            tail.append(line)
    results["seconds"][label] = round(time.time() - start, 1)
    detail = f"{results['seconds'][label]}s" if proc.returncode == 0 else "".join(tail)[-1500:]
    check(label, proc.returncode == 0, detail)
    return proc.returncode == 0


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


gpu = torch.cuda.is_available()
results["environment"] = {
    "gpu": torch.cuda.get_device_name(0) if gpu else "none",
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if gpu else 0,
    "cpu_cores": os.cpu_count(),
    "torch": torch.__version__,
    "misfit": misfit.__version__,
    "bf16": hardware.bf16_supported(),
}
print(json.dumps(results["environment"], indent=2))
check("GPU available", gpu)

## Download the data and checkpoints

About 4.7 GB in total. `._*` files in the MSD archive are macOS metadata, not
scans, so they are left out.

In [ ]:
if not Path("Task09_Spleen").exists():
    subprocess.run(["wget", "-q", "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar"], check=True)
    subprocess.run(["tar", "-xf", "Task09_Spleen.tar"], check=True)

scans = sorted(
    str(p.resolve())
    for folder in ("imagesTr", "imagesTs")
    for p in Path("Task09_Spleen", folder).glob("*.nii.gz")
    if not p.name.startswith(".")
)
print(f"{len(scans)} scans")

for name, spec in MODELS.items():
    spec["checkpoint"] = WORK / f"{name}.pth"
    if not spec["checkpoint"].exists():
        subprocess.run(["wget", "-q", spec["url"], "-O", str(spec["checkpoint"])], check=True)
    print(name, f"{spec['checkpoint'].stat().st_size / 2**30:.2f} GB")

pd.DataFrame({"path": scans}).to_csv(WORK / "paths.csv", index=False)
run("misfit_index", "index", "--input", WORK / "paths.csv", "--output", INDEX, "--num-workers-index", os.cpu_count() or 1)
index = pd.read_parquet(INDEX)
results["splits"] = index["split"].value_counts().to_dict()
print(results["splits"])
check("index has a val split", results["splits"].get("val", 0) > 0, results["splits"])

## 1. Memory and speed

Full training steps on random data: forward, backward, and AdamW, with the same
BF16 setting and masking the commands use. `OOM` means that combination doesn't
fit on this GPU.

In [ ]:
amp = hardware.resolve_amp(True, warn=False)


def probe(model_name, patch, batch_size):
    """Return (seconds per step, peak GPU GB) for real training steps."""
    device = torch.device("cuda" if gpu else "cpu")
    mask_patch_size = 8 if model_name.startswith("primus") else 16
    model = get_model_from_registry(
        model_name, in_channels=1, img_size=(patch,) * 3, mask_patch_size=mask_patch_size,
        mask_ratio=0.75, target_spacing=None,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.randn(batch_size, 1, patch, patch, patch, device=device)
    if gpu:
        torch.cuda.reset_peak_memory_stats()
    timed = 0.0
    for step in range(PROBE_STEPS + 1):
        if gpu:
            torch.cuda.synchronize()
        start = time.time()
        optimizer.zero_grad(set_to_none=True)
        with hardware.autocast_context(amp):
            out = model(x)
            loss = (((out["reconstruction"] - x) ** 2) * out["mask"]).sum() / out["mask"].sum()
        loss.backward()
        optimizer.step()
        if gpu:
            torch.cuda.synchronize()
        if step > 0:  # The first step includes setup.
            timed += time.time() - start
    peak = torch.cuda.max_memory_allocated() / 2**30 if gpu else 0.0
    return timed / PROBE_STEPS, peak


# A rerun in the same session reuses these results: the probe is the slowest part.
PROBE_CACHE = WORK / "probe.json"
planned = [] if PROBE_CACHE.exists() else PROBE
if PROBE_CACHE.exists():
    results["probe"] = json.loads(PROBE_CACHE.read_text())
    print(f"Reusing {PROBE_CACHE}; delete it to probe again.")

for model_name, patch in planned:
    for batch_size in PROBE_BATCH_SIZES:
        row = {"model": model_name, "patch": patch, "batch": batch_size}
        try:
            seconds, peak = probe(model_name, patch, batch_size)
            row.update(fits=True, s_per_step=round(seconds, 3), peak_gb=round(peak, 1),
                       samples_per_s=round(batch_size / seconds, 2))
        except torch.cuda.OutOfMemoryError:
            row.update(fits=False)
        results["probe"].append(row)
        print(row, flush=True)
        free_gpu()
        if not row["fits"]:
            break  # Larger batches won't fit either.

PROBE_CACHE.write_text(json.dumps(results["probe"]))
pd.DataFrame(results["probe"])

In [ ]:
# Use the largest probed batch size that fits, up to MAX_TRAIN_BATCH_SIZE.
for name, spec in MODELS.items():
    fits = [
        r["batch"] for r in results["probe"]
        if r["model"] == spec["model"] and r["patch"] == spec["patch"] and r["fits"]
        and r["batch"] <= MAX_TRAIN_BATCH_SIZE
    ]
    spec["batch"] = max(fits, default=0)
    check(f"{name} fits at {spec['patch']}^3", spec["batch"] > 0, f"training batch size {spec['batch']}")

## 2. Do the checkpoints load and work on real scans?

The closest match to how nnFoundation was pretrained: no `--target-spacing`
(native resolution), `z_score_only`, and `masked_mse`. Each checkpoint is scored
on the val split against the naive baseline and a randomly initialized network of
the same architecture, which is the control.

In [ ]:
def train_flags(spec):
    return [
        "--index", INDEX, "--model", spec["model"], "--patch-size", spec["patch"], spec["patch"], spec["patch"],
        "--mask-patch-size", spec["mask_patch_size"], "--mask-ratio", spec["mask_ratio"],
        "--normalization-policy", "z_score_only", "--loss", "masked_mse",
    ]


def evaluate(name, label, checkpoint, config):
    csv = WORK / f"eval_{name}_{label}.csv"
    if not run(f"evaluate {name} ({label})", "evaluate", "--checkpoint", checkpoint, "--index", INDEX,
               "--config", config, "--output-csv", csv, "--split", "val"):
        return None
    scores = pd.read_csv(csv).select_dtypes("number").mean()
    row = {"model": name, "weights": label, **{k: round(float(v), 4) for k, v in scores.items()}}
    results["evaluate"].append(row)
    return row


for name, spec in MODELS.items():
    spec["config"] = WORK / f"config_{name}" / "config.json"
    shutil.rmtree(spec["config"].parent, ignore_errors=True)
    run(f"write config {name}", "train", *train_flags(spec), "--results", spec["config"].parent, "--init-only")

    torch.manual_seed(0)
    model = get_model_from_registry(
        spec["model"], in_channels=1, img_size=(spec["patch"],) * 3, mask_patch_size=spec["mask_patch_size"],
        mask_ratio=spec["mask_ratio"], target_spacing=None,
    )
    spec["random"] = WORK / f"random_{name}.pt"
    torch.save({"model": model.state_dict()}, spec["random"])
    del model
    free_gpu()

    pretrained = evaluate(name, "pretrained", spec["checkpoint"], spec["config"])
    control = evaluate(name, "random", spec["random"], spec["config"])
    if pretrained and control:
        check(f"{name} beats the naive baseline", pretrained["masked_mse_skill"] > 0, pretrained["masked_mse_skill"])
        check(f"{name} beats a random network", pretrained["masked_mse_skill"] > control["masked_mse_skill"],
              f"{pretrained['masked_mse_skill']} vs {control['masked_mse_skill']}")

columns = ["model", "weights", "masked_mae", "masked_mse", "masked_mse_naive", "masked_mse_skill", "masked_psnr"]
pd.DataFrame(results["evaluate"], columns=columns)

## 3. Pretraining from `--init-weights` vs. random weights

The same short run twice for each model: starting from the checkpoint, and from
random weights. Validation loss in the first epochs shows the head start. This
schedule is far too short to say how a full pretraining run would end.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def losses(results_dir):
    events = EventAccumulator(str(Path(results_dir) / "logs"))
    events.Reload()
    return {tag: [round(s.value, 4) for s in events.Scalars(tag)] for tag in ("loss/train", "loss/val")}


def schedule(spec, epochs):
    return [
        "--batch-size", spec["batch"], "--epochs", epochs, "--steps-per-epoch", STEPS_PER_EPOCH,
        "--warmup-epochs", 1, "--val-volumes", 8, "--num-cpu-workers", CPU_WORKERS,
    ]


for name, spec in MODELS.items():
    if spec["batch"] == 0:
        continue
    results["training"][name] = {}
    for label, extra in (("pretrained", ["--init-weights", spec["checkpoint"]]), ("random", [])):
        out = WORK / f"run_{name}_{label}"
        shutil.rmtree(out, ignore_errors=True)
        run(f"train {name} ({label})", "train", *train_flags(spec), "--results", out, *schedule(spec, EPOCHS), *extra)
        results["training"][name][label] = losses(out)
        free_gpu()
    evaluate(name, "fine-tuned", WORK / f"run_{name}_pretrained" / "models" / "best_model.pt", spec["config"])
    evaluate(name, "random, trained", WORK / f"run_{name}_random" / "models" / "best_model.pt", spec["config"])

trained = list(results["training"])
fig, axes = plt.subplots(1, max(len(trained), 1), figsize=(6 * max(len(trained), 1), 4), squeeze=False)
for ax, name in zip(axes[0], trained, strict=False):
    for label, curves in results["training"][name].items():
        ax.plot(curves["loss/val"], marker="o", label=f"{label}: val")
        ax.plot(curves["loss/train"], linestyle="--", label=f"{label}: train")
    ax.set(title=name, xlabel="epoch", ylabel="masked MSE")
    ax.legend()
plt.tight_layout()
plt.show()
pd.DataFrame(results["evaluate"], columns=columns)

## 4. Resume, downstream commands, and MIST

In [ ]:
for name, spec in MODELS.items():
    if spec["batch"] == 0:
        continue
    out = WORK / f"run_{name}_pretrained"
    if run(f"resume {name}", "train", *train_flags(spec), "--results", out, *schedule(spec, EPOCHS + 1), "--resume"):
        epochs = len(losses(out)["loss/train"])
        check(f"{name} resumed through the extra epoch", epochs == EPOCHS + 1, f"{epochs} epochs logged")
    free_gpu()

    common = ["--index", INDEX, "--config", spec["config"], "--split", "val"]
    run(f"inspect {name}", "inspect", "--checkpoint", spec["checkpoint"], "--output-dir", WORK / f"inspect_{name}", *common)
    run(f"encode {name}", "encode", "--encoder-checkpoint", spec["checkpoint"], "--output-dir", WORK / f"encode_{name}", *common)
    run(f"embed {name}", "embed", "--encoder-checkpoint", spec["checkpoint"], "--output-dir", WORK / f"embed_{name}", *common)
    free_gpu()

    # These commands skip volumes that fail and still exit 0, so check their output.
    encoded_files = sorted((WORK / f"encode_{name}").glob("*.npz"))
    embedded_files = sorted((WORK / f"embed_{name}").glob("*.npz"))
    check(f"{name} encoded and embedded every val scan",
          len(encoded_files) == len(embedded_files) == results["splits"]["val"],
          f"{len(encoded_files)} and {len(embedded_files)} of {results['splits']['val']}")
    if not (encoded_files and embedded_files):
        continue
    encoded = np.load(encoded_files[0])
    embedded = np.load(embedded_files[0])
    spec_shapes = {"feature_map": encoded["feature_map"].shape, "embedding": embedded["embedding"].shape}
    results.setdefault("shapes", {})[name] = {k: list(v) for k, v in spec_shapes.items()}
    check(f"{name} features are finite",
          np.isfinite(encoded["feature_map"]).all() and np.isfinite(embedded["embedding"]).all(), spec_shapes)

In [ ]:
import mist.models  # noqa: E402, F401
from mist.models import model_loader
from mist.models.model_registry import get_model_from_registry as get_mist_model

for name, spec in MODELS.items():
    weights = WORK / f"run_{name}_pretrained" / "models" / "encoder_weights.pt"
    if not weights.exists():
        continue
    mist_model = get_mist_model(
        spec["model"], in_channels=1, out_channels=3, patch_size=[spec["patch"]] * 3, target_spacing=[1.0, 1.0, 1.0],
    )
    mist_model, summary = model_loader.load_pretrained_encoder(mist_model, str(weights))
    total = len(mist_model.get_encoder_state_dict())
    check(f"{name} fine-tuned encoder loads into MIST {spec['model']}",
          len(summary["loaded"]) == total and not summary["skipped"], f"{len(summary['loaded'])} of {total} tensors")
    del mist_model
    free_gpu()

### What the reconstructions look like

The middle axial slice of one val scan. Judge the **masked** regions: an MAE is
only trained there, so visible regions can look wrong without meaning anything.

In [ ]:
row = index[index["split"] == "val"].iloc[0]
scan = nib.load(row["path"]).get_fdata()
z = scan.shape[2] // 2
low, high = np.percentile(scan, [1, 99])

fig, axes = plt.subplots(len(MODELS), 3, figsize=(12, 4 * len(MODELS)), squeeze=False)
for axs, name in zip(axes, MODELS, strict=True):
    folder = WORK / f"inspect_{name}"
    recon_path = folder / "reconstructions" / f"{row['volume_id']}.nii.gz"
    check(f"{name} inspect wrote a reconstruction", recon_path.exists())
    if not recon_path.exists():
        continue
    recon = nib.load(recon_path).get_fdata()[:, :, z]
    mask = nib.load(folder / "masks" / f"{row['volume_id']}.nii.gz").get_fdata()[:, :, z]
    shown = {
        "input": scan[:, :, z],
        "input with masked regions blanked": np.where(mask > 0.5, low, scan[:, :, z]),
        "visible input + reconstructed masked regions": np.where(mask > 0.5, recon, scan[:, :, z]),
    }
    for ax, (title, image) in zip(axs, shown.items(), strict=True):
        ax.imshow(image.T, cmap="gray", origin="lower", vmin=low, vmax=high)
        ax.set_title(f"{name}: {title}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Results

Copy everything printed below `RESULTS` and paste it back.

In [ ]:
failed = [c["name"] for c in results["checks"] if not c["passed"]]
print(f"{len(results['checks']) - len(failed)} of {len(results['checks'])} checks passed")
for name in failed:
    print("FAILED:", name)
print("\nRESULTS")
print(json.dumps(results, indent=1, default=str))